In [1]:
import json
import pandas as pd

DATA_PATH = "C:\\Users\\USER\\Desktop\\MediRAG\\data division logic\\miriad_neurology.jsonl"

rows = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

df.head()

,qa_id,paper_id,question,answer,paper_url,paper_title,passage_text,passage_position,year,venue,specialty
0,38_17780367_0_1,17780367,What are the mechanisms underlying stroke in c...,The mechanisms underlying stroke in cancer pat...,https://api.semanticscholar.org/CorpusID:17780367,Clues to Occult Cancer in Patients with Ischem...,Cancer and cerebrovascular disease are the lea...,0,2012.0,PLoS ONE,Neurology
1,38_8335311_0_2,8335311,How is the volume of intracerebral hemorrhage ...,The volume of intracerebral hemorrhage is meas...,https://api.semanticscholar.org/CorpusID:8335311,Hemorrhage Prediction of 30-Day Mortality Amon...,Thrombolysis-Related Intracranial Hemorrhage I...,0,1998.0,Circulation,Neurology
2,38_8335311_0_3,8335311,How is brain edema defined and measured in pat...,Brain edema is defined as the low-density regi...,https://api.semanticscholar.org/CorpusID:8335311,Hemorrhage Prediction of 30-Day Mortality Amon...,Thrombolysis-Related Intracranial Hemorrhage I...,0,1998.0,Circulation,Neurology
3,38_8335311_1_1,8335311,What are some baseline clinical predictors of ...,Baseline clinical predictors of mortality in p...,https://api.semanticscholar.org/CorpusID:8335311,Hemorrhage Prediction of 30-Day Mortality Amon...,"Two independent investigators (M.A.S., K.W.M.)...",1,1998.0,Circulation,Neurology
4,38_8335311_1_2,8335311,How is the Glasgow Coma Scale score determined...,The Glasgow Coma Scale score in patients with ...,https://api.semanticscholar.org/CorpusID:8335311,Hemorrhage Prediction of 30-Day Mortality Amon...,"Two independent investigators (M.A.S., K.W.M.)...",1,1998.0,Circulation,Neurology


In [2]:
print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

Total rows: 205154
Columns: ['qa_id', 'paper_id', 'question', 'answer', 'paper_url', 'paper_title', 'passage_text', 'passage_position', 'year', 'venue', 'specialty']


In [3]:
# Split by specialty
neurology_df = df[df["specialty"].str.lower() == "neurology"]
neurosurgery_df = df[df["specialty"].str.lower() == "neurosurgery"]

print("Neurology rows:", len(neurology_df))
print("Neurosurgery rows:", len(neurosurgery_df))

Neurology rows: 198125
Neurosurgery rows: 7029


In [4]:
# Determine max balanced size
neurosurgery_count = len(neurosurgery_df)
print("Using per-specialty size:", neurosurgery_count)

# Sample Neurology to match Neurosurgery
neurology_sample = neurology_df.sample(
    n=neurosurgery_count,
    random_state=42
)

# Use ALL Neurosurgery rows
neurosurgery_sample = neurosurgery_df.copy()

# Combine and shuffle
df_subset = pd.concat(
    [neurology_sample, neurosurgery_sample],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print("Final subset size:", len(df_subset))
df_subset["specialty"].value_counts()

Using per-specialty size: 7029
Final subset size: 14058


specialty
Neurosurgery    7029
Neurology       7029
Name: count, dtype: int64

In [6]:
OUTPUT_PATH = "miriad_neuro_neurosurg_balanced_14k.jsonl"

df_subset.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"Saved subset to {OUTPUT_PATH}")

Saved subset to miriad_neuro_neurosurg_balanced_14k.jsonl


In [1]:
CONSTRAINT_SCHEMA = {
    "Patient_Demographics": [
        "Neonatal",
        "Pediatric",
        "Adult",
        "Geriatric",
        "Pregnant",
        "N/A"
    ],
    "Acuity_Phase": [
        "Acute",
        "Chronic",
        "Pre-op",
        "Post-op",
        "N/A"
    ],
    "Pathology": "FREE_TEXT",
    "Anatomy": "FREE_TEXT",
    "Modality_Procedure": "FREE_TEXT",
    "Negative_Constraint": "FREE_TEXT"
}

LABEL_DEFINITION = {
"PASS": "No hard constraint violation",
"FAIL": "One or more hard constraint violations"
}

In [2]:
TEACHER_PROMPT_TEMPLATE_V2 = """
You are a medical constraint and applicability verifier for a Neurology/Neurosurgery RAG system.

Your job is STRICT binary classification.

You must output FAIL if ANY of the following are true:

1) The Target Chunk explicitly contradicts a hard constraint in the query.
2) The Target Chunk discusses a different pathology than the one in the query.
3) The Target Chunk discusses a different anatomical region unrelated to the query.
4) The Target Chunk is clearly unrelated to answering the question.
5) The Target Chunk cannot reasonably help answer the query.

You must output PASS only if:
- The chunk is applicable to the query AND
- No hard constraint violations exist.

Important Rules:

- Do NOT judge answer completeness.
- Do NOT fail simply because details are missing.
- Only fail if the chunk is conflicting OR irrelevant.
- Medical topical mismatch = FAIL.
- Different condition/procedure/context = FAIL.

Constraint Schema:
- Patient_Demographics: {demographics}
- Acuity_Phase: {acuity}
- Pathology: FREE_TEXT
- Anatomy: FREE_TEXT
- Modality_Procedure: FREE_TEXT
- Negative_Constraint: FREE_TEXT

CRITICAL INSTRUCTIONS:

STEP 1:
Extract constraints ONLY from the User Query text.
Do NOT look at the Target Chunk when extracting constraints.

STEP 2:
After constraints are extracted from the User Query,
compare them against the Target Chunk.

Never copy terminology from the Target Chunk into the constraints field.
The constraints field must reflect the User Query only.

User Query:
\"\"\"{query}\"\"\"

Target Chunk:
\"\"\"{answer}\"\"\"

Output JSON ONLY:

{{
  "constraints": {{
    "Patient_Demographics": "... or N/A",
    "Acuity_Phase": "... or N/A",
    "Pathology": "... or N/A",
    "Anatomy": "... or N/A",
    "Modality_Procedure": "... or N/A",
    "Negative_Constraint": "... or N/A"
  }},
  "reasoning": "Brief explanation of why PASS or FAIL.",
  "label": "PASS or FAIL"
}}
"""

In [9]:
import json
import pandas as pd

SUBSET_PATH = "miriad_neuro_neurosurg_balanced_14k.jsonl"

rows = []
with open(SUBSET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df_subset = pd.DataFrame(rows)

print("Loaded rows:", len(df_subset))
df_subset["specialty"].value_counts()

Loaded rows: 14058


specialty
Neurosurgery    7029
Neurology       7029
Name: count, dtype: int64

In [10]:
pass_pairs = df_subset[["question", "answer"]].copy()
pass_pairs["pair_type"] = "PASS"

print("PASS pairs:", len(pass_pairs))
pass_pairs.head()

PASS pairs: 14058


,question,answer,pair_type
0,What are the advantages of the telovelar appro...,The telovelar approach offers early visualizat...,PASS
1,What are the factors to consider for a shunt o...,The factors to consider for a shunt operation ...,PASS
2,What factors have been reported as causes of t...,Recurrence of S-CSF-L from the clivus has been...,PASS
3,What are the potential complications and consi...,In surgical management of giant serpentine ane...,PASS
4,What challenges do neurosurgeons face in treat...,Neurosurgeons face challenges in treating pude...,PASS


In [11]:
FAIL_MULTIPLIER = 2
num_fail_pairs = len(pass_pairs) * FAIL_MULTIPLIER
print("Target FAIL pairs:", num_fail_pairs)

Target FAIL pairs: 28116


In [12]:
queries = df_subset[["qa_id", "question"]]
answers = df_subset[["qa_id", "answer"]]

In [13]:
import random

fail_pairs = []

while len(fail_pairs) < num_fail_pairs:
    q_row = queries.sample(1).iloc[0]
    a_row = answers.sample(1).iloc[0]

    # Ensure mismatch
    if q_row["qa_id"] == a_row["qa_id"]:
        continue

    fail_pairs.append({
        "question": q_row["question"],
        "answer": a_row["answer"],
        "pair_type": "FAIL"
    })

fail_pairs = pd.DataFrame(fail_pairs)

print("FAIL pairs:", len(fail_pairs))
fail_pairs.head()

FAIL pairs: 28116


,question,answer,pair_type
0,What are the factors that can affect the outco...,The surgical complications associated with pit...,FAIL
1,How does seizure control impact the health-rel...,The potential complications of anterior cervic...,FAIL
2,What are the indications for surgery in the ma...,Current therapeutic strategies for patients wi...,FAIL
3,What are the common complications associated w...,Several surgical approaches can be used for tu...,FAIL
4,How is the working distance depth affected by ...,The rationale for early decompression in cases...,FAIL


In [14]:
pair_candidates = pd.concat(
    [pass_pairs, fail_pairs],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print("Total candidate pairs:", len(pair_candidates))
pair_candidates["pair_type"].value_counts()

Total candidate pairs: 42174


pair_type
FAIL    28116
PASS    14058
Name: count, dtype: int64

In [15]:
OUTPUT_PATH = "slm1_pair_candidates.jsonl"

pair_candidates.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"Saved {len(pair_candidates)} candidate pairs to {OUTPUT_PATH}")

Saved 42174 candidate pairs to slm1_pair_candidates.jsonl


In [11]:
import json
import pandas as pd

CANDIDATE_PATH = "slm1_pair_candidates.jsonl"

rows = []
with open(CANDIDATE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

pairs_df = pd.DataFrame(rows)

print("Loaded candidate pairs:", len(pairs_df))
pairs_df["pair_type"].value_counts()

Loaded candidate pairs: 42174


pair_type
FAIL    28116
PASS    14058
Name: count, dtype: int64

In [12]:
import os

OUTPUT_PATH = "slm1_training_data_v2.jsonl"

# Resume support
already_done = 0
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        already_done = sum(1 for _ in f)

print("Already labeled rows:", already_done)

Already labeled rows: 13170


In [ ]:
#formats the teacher prompt template with actual values. values that you need to give each time to the labelling LLM.
def build_teacher_prompt_v2(query, answer):
    return TEACHER_PROMPT_TEMPLATE_V2.format(
        demographics=", ".join(CONSTRAINT_SCHEMA["Patient_Demographics"]),
        acuity=", ".join(CONSTRAINT_SCHEMA["Acuity_Phase"]),
        query=query,
        answer=answer
    )

In [14]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("OPENAI_API_KEY not found")

client = OpenAI(api_key=api_key)

print("OpenAI client ready for GPT-4o calls.")

OpenAI client ready for GPT-4o calls.


In [15]:
# Take a small mixed sample
probe_df = pairs_df.sample(n=10, random_state=42).reset_index(drop=True)

probe_df[["pair_type", "question", "answer"]]

,pair_type,question,answer
0,FAIL,What is the purpose of cranial decompressive s...,The treatment options for cerebral radionecros...
1,FAIL,What are some common techniques used for anter...,The necessity of antibiotic treatment for all ...
2,FAIL,How do serological levels of GFAP change after...,"After tumor resection, an external ventricular..."
3,PASS,How does the Spetzler-Martin classification sy...,The Spetzler-Martin classification system is u...
4,FAIL,How do surgeons mitigate the risk of cerebrosp...,Simultaneous EEG and PET studies can be used t...
5,PASS,How has the implementation of facial and vesti...,The implementation of facial and vestibulococh...
6,FAIL,What are the common causes of pneumocephalus a...,There are several surgical treatment options f...
7,FAIL,What are the clinical hallmarks and common sym...,-Synuclein plays a central role in the pathoge...
8,FAIL,How are abdominal complications of VP shunts t...,Nerve injury induces complex plasticity of Gal...
9,FAIL,How is foot drop diagnosed and what factors sh...,"Gender, age at disease onset, and the time to ..."


In [16]:
row = probe_df.iloc[0]

prompt = build_teacher_prompt_v2(
    query=row["question"],
    answer=row["answer"]
)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a strict JSON generator."},
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

raw_output = response.choices[0].message.content
print(raw_output)

{
  "constraints": {
    "Patient_Demographics": "N/A",
    "Acuity_Phase": "N/A",
    "Pathology": "cerebral radionecrosis",
    "Anatomy": "cerebral",
    "Modality_Procedure": "cranial decompressive surgery",
    "Negative_Constraint": "N/A"
  },
  "reasoning": "The Target Chunk discusses treatment options for cerebral radionecrosis, which is a different pathology than cranial decompressive surgery proposed by Cushing. Therefore, it is not applicable to the query.",
  "label": "FAIL"
}


In [17]:
import re
import json

def extract_json(text: str):
    """
    Safely extract JSON from a model response.
    Handles markdown fences and extra whitespace.
    """
    # Remove markdown fences if present
    text = text.strip()

    # Match ```json ... ``` or ``` ... ```
    fence_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence_match:
        text = fence_match.group(1)

    # Final attempt to parse
    return json.loads(text)

In [18]:
import json

parsed = extract_json(raw_output)

parsed

{'constraints': {'Patient_Demographics': 'N/A',
  'Acuity_Phase': 'N/A',
  'Pathology': 'cerebral radionecrosis',
  'Anatomy': 'cerebral',
  'Modality_Procedure': 'cranial decompressive surgery',
  'Negative_Constraint': 'N/A'},
 'reasoning': 'The Target Chunk discusses treatment options for cerebral radionecrosis, which is a different pathology than cranial decompressive surgery proposed by Cushing. Therefore, it is not applicable to the query.',
 'label': 'FAIL'}

In [19]:
probe_results = []

for _, row in probe_df.iterrows():
    prompt = build_teacher_prompt_v2(
        query=row["question"],
        answer=row["answer"]
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a strict JSON generator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    output = response.choices[0].message.content

    # IMPORTANT: use the safe extractor
    parsed = extract_json(output)

    probe_results.append({
        "query": row["question"],
        "target_chunk": row["answer"],
        "constraints": parsed["constraints"],
        "label": parsed["label"],
        "pair_type": row["pair_type"]  # for sanity check only
    })

In [20]:
import json

PROBE_OUTPUT_PATH = "slm1_probe_results.json"

with open(PROBE_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(probe_results, f, ensure_ascii=False, indent=2)

print(f"Saved probe results to {PROBE_OUTPUT_PATH}")

Saved probe results to slm1_probe_results.json


In [21]:
import json
import pandas as pd
import os
import time
from tqdm import tqdm

CANDIDATE_PATH = "slm1_pair_candidates.jsonl"
OUTPUT_PATH = "slm1_training_data_v2.jsonl"

rows = []
with open(CANDIDATE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

pairs_df = pd.DataFrame(rows)
print("Total candidate pairs:", len(pairs_df))

Total candidate pairs: 42174


In [22]:
already_done = 0
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        already_done = sum(1 for _ in f)

print("Already labeled rows:", already_done)

Already labeled rows: 13170


In [23]:
BATCH_SIZE = 50

def batch_iterator(df, start, batch_size):
    for i in range(start, len(df), batch_size):
        yield i, df.iloc[i:i + batch_size]

In [24]:

MAX_ROWS = 19000  # hard safety cap

written = already_done  # how many rows are already in OUTPUT_PATH

with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
    for start_idx, batch in batch_iterator(pairs_df, already_done, BATCH_SIZE):

        # Batch-level safety check
        if written >= MAX_ROWS:
            print(f"\n🛑 Reached MAX_ROWS = {MAX_ROWS}. Stopping safely.")
            break

        print(f"\nProcessing rows {start_idx} → {start_idx + len(batch)}")

        for _, row in tqdm(batch.iterrows(), total=len(batch)):

            # Row-level safety check
            if written >= MAX_ROWS:
                print(f"\n🛑 Reached MAX_ROWS = {MAX_ROWS}. Stopping safely.")
                break

            prompt = build_teacher_prompt_v2(
                query=row["question"],
                answer=row["answer"]
            )

            try:
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": "You are a strict JSON generator."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0
                )

                output = response.choices[0].message.content
                parsed = extract_json(output)

                record = {
                    "query": row["question"],
                    "target_chunk": row["answer"],
                    "constraints": parsed.get("constraints", {}),
                    "reasoning": parsed.get("reasoning", ""),
                    "label": parsed.get("label", "UNKNOWN")
                }

                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                written += 1  # ✅ increment only after successful write

            except Exception as e:
                print("⚠️ Error on row, skipping:", e)
                continue

        # Gentle rate-limit buffer between batches
        time.sleep(1)


Processing rows 13170 → 13220


100%|██████████| 50/50 [02:52<00:00,  3.46s/it]



Processing rows 13220 → 13270


100%|██████████| 50/50 [02:37<00:00,  3.15s/it]



Processing rows 13270 → 13320


...


Processing rows 18920 → 18970


100%|██████████| 50/50 [02:41<00:00,  3.22s/it]



Processing rows 18970 → 19020


 60%|██████    | 30/50 [01:41<01:07,  3.37s/it]



🛑 Reached MAX_ROWS = 19000. Stopping safely.

🛑 Reached MAX_ROWS = 19000. Stopping safely.


In [1]:
import json
import pandas as pd

DATA_PATH = "slm1_training_data_v2.jsonl"

rows = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

print("Total rows:", len(df))
df["label"].value_counts()

Total rows: 19000


label
FAIL    12419
PASS     6581
Name: count, dtype: int64

In [2]:
before = len(df)

df = df.drop_duplicates(
    subset=["query", "target_chunk", "label"]
).reset_index(drop=True)

after = len(df)

print(f"Removed {before - after} duplicate rows")
print("Remaining rows:", after)

Removed 0 duplicate rows
Remaining rows: 19000


In [3]:
df["label"].value_counts(normalize=True)

label
FAIL    0.653632
PASS    0.346368
Name: proportion, dtype: float64

In [ ]:
# def build_slm_input(row):
#     return f"""You are a medical constraint verification model.

# Query:
# {row['query']}

# Target Chunk:
# {row['target_chunk']}

# Constraints:
# {json.dumps(row['constraints'], ensure_ascii=False)}
# """


In [ ]:
# print(build_slm_input(df.iloc[0]))
# print("Label:", df.iloc[0]["label"])

You are a medical constraint verification model.

Query:
What is the relationship between the extent of white matter damage and the severity of traumatic brain injuries?

Target Chunk:
Lactate, which was historically considered an end waste product, is now recognized as an important fuel and plays a role in neuroenergetic metabolism. In the setting of TBI, lactate serves as a preferential substitute fuel alongside glucose. HSL not only reduces ICP but also serves as an exogenous supplementation of neuroenergetic fuel, making it a potential neuroprotective fluid. Further evidence is needed to confirm these findings, but HSL may serve as an alternative fluid of choice in managing intracranial hypertension in TBI patients.

Constraints:
{"Patient_Demographics": "N/A", "Acuity_Phase": "N/A", "Pathology": "traumatic brain injuries", "Anatomy": "white matter", "Modality_Procedure": "N/A", "Negative_Constraint": "N/A"}

Label: PASS


In [ ]:
# FINAL_OUTPUT = "slm1_finetune_dataset.jsonl"

# with open(FINAL_OUTPUT, "w", encoding="utf-8") as f:
#     for _, row in df.iterrows():
#         record = {
#             "messages": [
#                 {
#                     "role": "system",
#                     "content": "You are a medical constraint verification model."
#                 },
#                 {
#                     "role": "user",
#                     "content": build_slm_input(row)
#                 },
#                 {
#                     "role": "assistant",
#                     "content": row["label"]
#                 }
#             ]
#         }
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")

# print("Saved fine-tuning dataset to:", FINAL_OUTPUT)

Saved fine-tuning dataset to: slm1_finetune_dataset.jsonl


In [ ]:
# # quick validation
# import random

# with open(FINAL_OUTPUT, "r", encoding="utf-8") as f:
#     sample = random.sample(list(f), 3)

# for s in sample:
#     print(json.dumps(json.loads(s), indent=2))

{
  "messages": [
    {
      "role": "system",
      "content": "You are a medical constraint verification model."
    },
    {
      "role": "user",
      "content": "You are a medical constraint verification model.\n\nQuery:\nWhat is the recommended surgical approach for the management of RDD in the central nervous system (CNS)?\n\n\nTarget Chunk:\nIn cases of primary CNS RDD, radical resection is the ideal surgical goal. However, in many cases, the mass tightly adheres to the cerebral cortex and may invade or surround other critical structures, making complete resection difficult. Adjunctive irradiation has been used in some patients with variable efficiency. Chemotherapy regimens have generally yielded negative results, although a combination of methotrexate and mercaptopurine has shown a sustained response in some children. Interferon-a-2a, particularly the pegylated form, has shown dramatic efficacy in cases of systemic RDD, but in the case described, azathioprine was preferred 

In [6]:
SYSTEM_PROMPT = """You are a medical constraint-aware verifier for a Neurology/Neurosurgery RAG system.

Your job is STRICT binary classification.

You must output FAIL if ANY of the following are true:

1) The Target Chunk explicitly contradicts a hard constraint in the query.
2) The Target Chunk discusses a different pathology than the one in the query.
3) The Target Chunk discusses a different anatomical region unrelated to the query.
4) The Target Chunk is clearly unrelated to answering the question.
5) The Target Chunk cannot reasonably help answer the query.

You must output PASS only if:
- The chunk is applicable to the query AND
- No hard constraint violations exist.

Output STRICT JSON in this format:

{
  "constraints": {...},
  "reasoning": "...",
  "label": "PASS or FAIL"
}
"""

In [7]:
import json
from tqdm import tqdm

INPUT_PATH = "slm1_training_data_v2.jsonl"   # your regenerated dataset
OUTPUT_PATH = "slm1_finetune_chat.jsonl"

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

with open(OUTPUT_PATH, "w", encoding="utf-8") as out_f:
    for row in tqdm(rows):

        user_content = f"""Query:
{row['query']}

Target Chunk:
{row['target_chunk']}"""

        assistant_output = {
            "constraints": row["constraints"],
            "reasoning": row["reasoning"],
            "label": row["label"]
        }

        chat_example = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": json.dumps(assistant_output, ensure_ascii=False)}
            ]
        }

        out_f.write(json.dumps(chat_example, ensure_ascii=False) + "\n")

print("✅ Conversion complete.")

100%|██████████| 19000/19000 [00:00<00:00, 28142.18it/s]

✅ Conversion complete.
